In [1]:
# Exercise 1: Conway's Game of Life

class Cell:
    """
    Represents a single cell in the Game of Life grid.
    A cell can be either alive (True) or dead (False).
    """
    def __init__(self, is_alive=False):
        """
        Initializes a Cell.

        Args:
            is_alive (bool): The initial state of the cell (True for alive, False for dead).
        """
        self.is_alive = is_alive

    def __repr__(self):
        """
        Returns a string representation of the cell for debugging.
        """
        return "1" if self.is_alive else "0"

    def display(self):
        """
        Returns the character to display for the cell's state.
        '█' for alive, ' ' for dead.
        """
        return '█' if self.is_alive else ' '

class GameOfLife:
    """
    Implements Conway's Game of Life on a two-dimensional grid with fixed borders.
    """
    def __init__(self, rows, cols, initial_state=None):
        """
        Initializes the Game of Life grid.

        Args:
            rows (int): The number of rows in the grid.
            cols (int): The number of columns in the grid.
            initial_state (list of lists, optional): A 2D list representing the
                                                     initial state of live (1) and dead (0) cells.
                                                     If None, the grid starts all dead.
        Raises:
            ValueError: If initial_state dimensions do not match rows/cols.
        """
        if rows <= 0 or cols <= 0:
            raise ValueError("Grid dimensions must be positive.")

        self.rows = rows
        self.cols = cols
        self.grid = []

        # Initialize the grid with Cell objects
        for r in range(rows):
            row_cells = []
            for c in range(cols):
                if initial_state and r < len(initial_state) and c < len(initial_state[r]):
                    # If initial_state is provided, set cell state based on it
                    row_cells.append(Cell(is_alive=bool(initial_state[r][c])))
                else:
                    # Otherwise, all cells start dead
                    row_cells.append(Cell(is_alive=False))
            self.grid.append(row_cells)

        print(f"Game of Life grid initialized: {self.rows}x{self.cols}")

    def display_grid(self, generation):
        """
        Prints the current state of the grid to the console.
        """
        print(f"\n--- Generation {generation} ---")
        print("+" + "-" * (self.cols * 2 + 1) + "+") # Top border
        for r in range(self.rows):
            row_display = "|"
            for c in range(self.cols):
                row_display += self.grid[r][c].display() + " " # Cell char + space for spacing
            row_display += "|"
            print(row_display)
        print("+" + "-" * (self.cols * 2 + 1) + "+") # Bottom border

    def _get_live_neighbors_count(self, row, col):
        """
        Counts the number of live neighbors for a given cell (row, col).
        Considers fixed borders: cells outside the grid are treated as dead.
        """
        live_neighbors = 0
        # Iterate over all 8 possible neighbor positions
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if dr == 0 and dc == 0:  # Skip the cell itself
                    continue

                neighbor_row, neighbor_col = row + dr, col + dc

                # Check if the neighbor is within grid boundaries
                if 0 <= neighbor_row < self.rows and 0 <= neighbor_col < self.cols:
                    if self.grid[neighbor_row][neighbor_col].is_alive:
                        live_neighbors += 1
        return live_neighbors

    def next_generation(self):
        """
        Applies the Game of Life rules to compute the next state of the grid.
        Returns True if the grid state changed, False if it became stable.
        """
        new_grid_state = []
        for r in range(self.rows):
            new_row = []
            for c in range(self.cols):
                cell = self.grid[r][c]
                live_neighbors = self._get_live_neighbors_count(r, c)

                new_cell_state = cell.is_alive # Assume state remains same unless rule applies

                # Rule 1: Any live cell with fewer than two live neighbours dies (underpopulation).
                if cell.is_alive and live_neighbors < 2:
                    new_cell_state = False
                # Rule 2: Any live cell with two or three live neighbours lives on to the next generation.
                # (No change needed, new_cell_state remains True)
                # Rule 3: Any live cell with more than three live neighbours dies (overpopulation).
                elif cell.is_alive and live_neighbors > 3:
                    new_cell_state = False
                # Rule 4: Any dead cell with exactly three live neighbours becomes a live cell (reproduction).
                elif not cell.is_alive and live_neighbors == 3:
                    new_cell_state = True

                new_row.append(Cell(is_alive=new_cell_state))
            new_grid_state.append(new_row)

        # Check if the grid has changed (for stability detection)
        grid_changed = False
        for r in range(self.rows):
            for c in range(self.cols):
                if self.grid[r][c].is_alive != new_grid_state[r][c].is_alive:
                    grid_changed = True
                    break
            if grid_changed:
                break

        self.grid = new_grid_state # Update the grid to the new state
        return grid_changed

    def run_game(self, max_generations=100, stable_generations_threshold=5):
        """
        Runs the Game of Life simulation for a specified number of generations
        or until the grid becomes stable (no changes for a few generations).

        Args:
            max_generations (int): The maximum number of generations to simulate.
            stable_generations_threshold (int): Number of consecutive generations
                                                with no change to consider the game stable.
        """
        print("\n--- Starting Game of Life Simulation ---")
        consecutive_stable_generations = 0

        for generation in range(max_generations):
            self.display_grid(generation)
            grid_changed = self.next_generation()

            if not grid_changed:
                consecutive_stable_generations += 1
                if consecutive_stable_generations >= stable_generations_threshold:
                    print(f"\nGame became stable after {generation + 1} generations.")
                    self.display_grid(generation + 1) # Display final stable state
                    break
            else:
                consecutive_stable_generations = 0 # Reset if grid changed

        else: # This 'else' block executes if the loop completes without a 'break'
            print(f"\nSimulation ended after {max_generations} generations (max generations reached).")
            self.display_grid(max_generations) # Display final state

# --- Initial States for Testing ---
# 0 represents a dead cell, 1 represents a live cell

# 1. Blinker (period 2 oscillator)
blinker_initial_state = [
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
]

# 2. Glider (moves across the grid)
glider_initial_state = [
    [0, 1, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 1, 1, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
]

# 3. Toad (period 2 oscillator)
toad_initial_state = [
    [0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 1, 0],
    [0, 1, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0],
]

# 4. Beacon (period 2 oscillator)
beacon_initial_state = [
    [0, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 0, 0],
    [0, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0],
    [0, 0, 0, 1, 1, 0],
    [0, 0, 0, 0, 0, 0],
]

# 5. R-pentomino (complex, takes many generations to stabilize)
r_pentomino_initial_state = [
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
]


# --- Main execution block ---
if __name__ == "__main__":
    print("--- Running Game of Life Simulations ---")

    # Test 1: Blinker
    print("\n\n--- Simulation: Blinker (5x5 grid) ---")
    game1 = GameOfLife(5, 5, blinker_initial_state)
    game1.run_game(max_generations=10) # Blinker is stable in 2 generations

    # Test 2: Glider
    print("\n\n--- Simulation: Glider (10x10 grid) ---")
    game2 = GameOfLife(10, 10, glider_initial_state)
    game2.run_game(max_generations=20) # Glider moves off the fixed grid

    # Test 3: Toad
    print("\n\n--- Simulation: Toad (6x6 grid) ---")
    game3 = GameOfLife(6, 6, toad_initial_state)
    game3.run_game(max_generations=10) # Toad is stable in 2 generations

    # Test 4: Beacon
    print("\n\n--- Simulation: Beacon (6x6 grid) ---")
    game4 = GameOfLife(6, 6, beacon_initial_state)
    game4.run_game(max_generations=10) # Beacon is stable in 2 generations

    # Test 5: R-pentomino (more complex, takes longer to stabilize)
    print("\n\n--- Simulation: R-pentomino (30x30 grid) ---")
    game5 = GameOfLife(30, 30, r_pentomino_initial_state)
    game5.run_game(max_generations=100) # This will run for 100 generations or until stable

    print("\nAll Game of Life simulations completed.")


--- Running Game of Life Simulations ---


--- Simulation: Blinker (5x5 grid) ---
Game of Life grid initialized: 5x5

--- Starting Game of Life Simulation ---

--- Generation 0 ---
+-----------+
|          |
|          |
|  █ █ █   |
|          |
|          |
+-----------+

--- Generation 1 ---
+-----------+
|          |
|    █     |
|    █     |
|    █     |
|          |
+-----------+

--- Generation 2 ---
+-----------+
|          |
|          |
|  █ █ █   |
|          |
|          |
+-----------+

--- Generation 3 ---
+-----------+
|          |
|    █     |
|    █     |
|    █     |
|          |
+-----------+

--- Generation 4 ---
+-----------+
|          |
|          |
|  █ █ █   |
|          |
|          |
+-----------+

--- Generation 5 ---
+-----------+
|          |
|    █     |
|    █     |
|    █     |
|          |
+-----------+

--- Generation 6 ---
+-----------+
|          |
|          |
|  █ █ █   |
|          |
|          |
+-----------+

--- Generation 7 ---
+-----------+
